# 03 — Feature Engineering

Application-level derived features (ages, ratios, bands) plus customer-level
aggregates built from every related table (bureau, bureau_balance,
previous_application, POS_CASH_balance, installments_payments,
credit_card_balance). This mirrors `utils/feature_engineering.py`, which the
Streamlit pages call directly.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append("..")

from utils.preprocessing import clean_application_data
from utils.feature_engineering import (
    add_application_features, aggregate_bureau_features, aggregate_bureau_balance_features,
    aggregate_previous_application_features, aggregate_pos_cash_features,
    add_installment_features, aggregate_installment_features,
    add_credit_card_features, aggregate_credit_card_features,
    build_master_customer_table, assign_risk_segment,
)

app = pd.read_csv("../data/application_train.csv")
bureau = pd.read_csv("../data/bureau.csv")
bureau_balance = pd.read_csv("../data/bureau_balance.csv")
previous_application = pd.read_csv("../data/previous_application.csv")
pos_cash = pd.read_csv("../data/POS_CASH_balance.csv")
installments = pd.read_csv("../data/installments_payments.csv")
credit_card = pd.read_csv("../data/credit_card_balance.csv")

app_clean = clean_application_data(app)

## Application-level features

`AGE_YEARS`, `EMPLOYMENT_YEARS`, `INCOME_GROUP`, `AGE_GROUP`, `EMPLOYMENT_GROUP`, `CREDIT_TO_INCOME`, `ANNUITY_TO_INCOME`, `GOODS_TO_INCOME`, `CREDIT_TO_GOODS`, `INCOME_PER_FAMILY_MEMBER`.

In [2]:
app_feat = add_application_features(app_clean)
new_cols = [c for c in app_feat.columns if c not in app_clean.columns]
print(f"{len(new_cols)} new columns:")
new_cols

16 new columns:


['AGE_YEARS',
 'AGE_GROUP',
 'EMPLOYMENT_YEARS',
 'EMPLOYMENT_GROUP',
 'INCOME_GROUP',
 'INCOME_PERCENTILE',
 'INCOME_PER_FAMILY_MEMBER',
 'INCOME_PER_CHILD',
 'CREDIT_TO_INCOME',
 'ANNUITY_TO_INCOME',
 'GOODS_TO_INCOME',
 'CREDIT_TO_GOODS',
 'CREDIT_BAND',
 'CREDIT_TO_INCOME_BAND',
 'ANNUITY_TO_INCOME_BAND',
 'REPAYMENT_STATUS']

In [3]:
app_feat[["AGE_YEARS", "AGE_GROUP", "EMPLOYMENT_YEARS", "EMPLOYMENT_GROUP",
          "INCOME_GROUP", "CREDIT_TO_INCOME", "ANNUITY_TO_INCOME", "GOODS_TO_INCOME",
          "CREDIT_TO_GOODS", "INCOME_PER_FAMILY_MEMBER"]].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
AGE_YEARS,1000.0,NaN,NaN,NaN,43.4851,11.606634,21.1,33.8,43.1,52.7,68.8
AGE_GROUP,1000,5,31-40,270,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EMPLOYMENT_YEARS,842.0,NaN,NaN,NaN,6.422328,6.489018,0.0,2.0,4.25,8.6,42.8
EMPLOYMENT_GROUP,842,6,1-3 Years,198,NaN,NaN,NaN,NaN,NaN,NaN,NaN
INCOME_GROUP,1000,5,Very Low,200,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CREDIT_TO_INCOME,1000.0,NaN,NaN,NaN,3.92445,2.651983,0.29,2.02,3.18,5.2925,34.92
ANNUITY_TO_INCOME,1000.0,NaN,NaN,NaN,0.180779,0.098988,0.014,0.116,0.161,0.228,1.374
GOODS_TO_INCOME,999.0,NaN,NaN,NaN,3.518819,2.434516,0.29,1.85,2.92,4.585,34.92
CREDIT_TO_GOODS,999.0,NaN,NaN,NaN,1.122973,0.121808,1.0,1.0,1.12,1.2,1.66
INCOME_PER_FAMILY_MEMBER,1000.0,NaN,NaN,NaN,94052.27383,69648.159054,10800.0,49500.0,76500.0,112500.0,765000.0


## Bureau aggregates → customer level

`BUREAU_ACCOUNT_COUNT`, `ACTIVE_BUREAU_COUNT`, `CLOSED_BUREAU_COUNT`, `TOTAL_BUREAU_CREDIT`, `TOTAL_BUREAU_DEBT`, `AVG_BUREAU_CREDIT`, `TOTAL_BUREAU_OVERDUE`, `MAX_BUREAU_OVERDUE`.

In [4]:
bureau_agg = aggregate_bureau_features(bureau)
print(bureau_agg.shape)
bureau_agg.head()

(205, 9)


,SK_ID_CURR,BUREAU_ACCOUNT_COUNT,TOTAL_BUREAU_CREDIT,AVG_BUREAU_CREDIT,TOTAL_BUREAU_DEBT,TOTAL_BUREAU_OVERDUE,MAX_BUREAU_OVERDUE,ACTIVE_BUREAU_COUNT,CLOSED_BUREAU_COUNT
0,101060,4,816575.490,204143.872500,408874.32,0.0,0.000,2,2
1,104261,1,450000.000,450000.000000,317925.00,0.0,NaN,1,0
2,115001,8,1693647.900,211705.987500,521973.00,0.0,0.000,2,6
3,118247,7,548801.775,78400.253571,0.00,0.0,8850.375,2,5
4,119306,3,290295.000,96765.000000,63688.50,0.0,0.000,1,2


## Bureau balance aggregates → customer level

Joins `bureau_balance` through `bureau` (on `SK_ID_BUREAU`) to reach `SK_ID_CURR`, then builds `MONTHS_WITH_DELINQUENCY`, `MAX_DELINQUENCY_LEVEL`, `CLOSED_MONTHS_COUNT`, `ACTIVE_MONTHS_COUNT`.

In [5]:
bb_agg = aggregate_bureau_balance_features(bureau_balance, bureau)
print(bb_agg.shape)
bb_agg.head()

(6, 6)


,SK_ID_CURR,BUREAU_BALANCE_MONTHS,MONTHS_WITH_DELINQUENCY,MAX_DELINQUENCY_LEVEL,CLOSED_MONTHS_COUNT,ACTIVE_MONTHS_COUNT
0,357810,31,0,0,0,31
1,380361,306,0,0,97,209
2,393321,22,0,0,0,22
3,400434,33,0,0,21,12
4,401081,42,0,0,25,17


## Previous application aggregates → customer level

`PREVIOUS_APPLICATION_COUNT`, `PREVIOUS_APPROVED_COUNT`, `PREVIOUS_REFUSED_COUNT`, `PREVIOUS_APPROVAL_RATE`, `AVG_PREVIOUS_CREDIT`, `MAX_PREVIOUS_CREDIT`.

In [6]:
prev_agg = aggregate_previous_application_features(previous_application)
print(prev_agg.shape)
prev_agg.head()

(985, 7)


,SK_ID_CURR,PREVIOUS_APPLICATION_COUNT,AVG_PREVIOUS_CREDIT,MAX_PREVIOUS_CREDIT,PREVIOUS_APPROVED_COUNT,PREVIOUS_REFUSED_COUNT,PREVIOUS_APPROVAL_RATE
0,100077,1,0.0,0.0,0,1,0.0
1,100373,1,115119.0,115119.0,1,0,100.0
2,101011,1,66735.0,66735.0,1,0,100.0
3,101167,1,120060.0,120060.0,1,0,100.0
4,101211,1,21204.0,21204.0,1,0,100.0


## POS/CASH aggregates → customer level

`AVG_DPD`, `MAX_DPD`, `TOTAL_DPD_EVENTS`, `AVG_INSTALMENTS_REMAINING`, `COMPLETED_CONTRACT_COUNT`.

In [7]:
pos_agg = aggregate_pos_cash_features(pos_cash)
print(pos_agg.shape)
pos_agg.head()

(989, 6)


,SK_ID_CURR,AVG_DPD,MAX_DPD,AVG_INSTALMENTS_REMAINING,TOTAL_DPD_EVENTS,COMPLETED_CONTRACT_COUNT
0,100187,0.0,0,8.0,0,0
1,100457,0.0,0,23.0,0,0
2,100676,0.0,0,12.0,0,0
3,101741,0.0,0,8.0,0,0
4,102489,0.0,0,22.0,0,0


## Installment payment features → customer level

Row-level: `PAYMENT_DELAY = DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT`,
`PAYMENT_DIFFERENCE = AMT_PAYMENT - AMT_INSTALMENT`,
`PAYMENT_RATIO = AMT_PAYMENT / AMT_INSTALMENT`, classified into
Early/On-Time/Late (timing) and Underpayment/Full/Overpayment (amount).

Customer-level: `TOTAL_INSTALLMENTS`, `LATE_PAYMENT_COUNT`, `LATE_PAYMENT_PERCENTAGE`,
`AVG_PAYMENT_DELAY`, `MAX_PAYMENT_DELAY`, `AVG_PAYMENT_RATIO`, `UNDERPAYMENT_COUNT`.

In [8]:
inst_feat = add_installment_features(installments)
inst_feat[["PAYMENT_DELAY", "PAYMENT_DIFFERENCE", "PAYMENT_RATIO", "PAYMENT_CLASS"]].head()

,PAYMENT_DELAY,PAYMENT_DIFFERENCE,PAYMENT_RATIO,PAYMENT_CLASS
0,-7.0,0.000,1.000,Early Payment
1,0.0,0.000,1.000,On-Time Payment
2,0.0,0.000,1.000,On-Time Payment
3,-8.0,0.000,1.000,Early Payment
4,17.0,-4.455,0.998,Late Payment


In [9]:
inst_agg = aggregate_installment_features(inst_feat)
print(inst_agg.shape)
inst_agg.head()

(965, 8)


,SK_ID_CURR,TOTAL_INSTALLMENTS,AVG_PAYMENT_DELAY,MAX_PAYMENT_DELAY,AVG_PAYMENT_RATIO,LATE_PAYMENT_COUNT,LATE_PAYMENT_PERCENTAGE,UNDERPAYMENT_COUNT
0,100012,1,-41.0,-41.0,1.0,0,0.0,0
1,100187,1,-1.0,-1.0,1.0,0,0.0,0
2,100193,1,0.0,0.0,1.0,0,0.0,0
3,100266,1,0.0,0.0,1.0,0,0.0,0
4,100299,1,30.0,30.0,0.0,1,100.0,1


## Credit card balance aggregates → customer level

Row-level: `CREDIT_UTILIZATION = AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL`.

Customer-level: `AVG_CC_BALANCE`, `MAX_CC_BALANCE`, `AVG_CC_LIMIT`, `AVG_CC_UTILIZATION`, `MAX_CC_UTILIZATION`, `TOTAL_CC_DRAWINGS`, `AVG_CC_PAYMENT`, `MAX_CC_DPD`.

In [10]:
cc_feat = add_credit_card_features(credit_card)
cc_agg = aggregate_credit_card_features(cc_feat)
print(cc_agg.shape)
cc_agg.head()

(1000, 9)


,SK_ID_CURR,AVG_CC_BALANCE,MAX_CC_BALANCE,AVG_CC_LIMIT,AVG_CC_UTILIZATION,MAX_CC_UTILIZATION,TOTAL_CC_DRAWINGS,AVG_CC_PAYMENT,MAX_CC_DPD
0,100218,748180.935,748180.935,720000.0,1.039,1.039,21073.905,45000.000,0
1,100662,0.000,0.000,0.0,NaN,NaN,0.000,NaN,0
2,100796,0.000,0.000,292500.0,0.000,0.000,0.000,509.175,0
3,101399,410062.365,410062.365,450000.0,0.911,0.911,2625.615,22500.000,0
4,101827,0.000,0.000,135000.0,0.000,0.000,0.000,NaN,0


## Master customer table + rule-based risk segmentation

Left-joins every aggregate above onto the application table (one row per `SK_ID_CURR`), then assigns a descriptive risk segment — NOT a prediction — from five elevated-risk signals.

In [11]:
master = build_master_customer_table(app_feat, bureau_agg, prev_agg, pos_agg, inst_agg, cc_agg)
master = master.merge(bb_agg, on="SK_ID_CURR", how="left")
master["RISK_SEGMENT"] = assign_risk_segment(master)
print(master.shape)
master["RISK_SEGMENT"].value_counts()

(1000, 178)


RISK_SEGMENT
Low Observed Risk         713
Moderate Observed Risk    204
Elevated Observed Risk     83
High Observed Risk          0
Name: count, dtype: int64

In [12]:
master.groupby("RISK_SEGMENT", observed=True)["TARGET"].mean() * 100  # validation check only, not used to build the segments

RISK_SEGMENT
Low Observed Risk         6.732118
Moderate Observed Risk    8.333333
Elevated Observed Risk    6.024096
Name: TARGET, dtype: float64

## Next notebook
04_eda.ipynb performs univariate, bivariate, and multivariate exploratory analysis on the engineered dataset.